# Prompt Engineering & Fine-Tuning
Hands-on exploration of prompt engineering techniques and fine-tuning concepts using real transformer models.

**Prerequisites:**
```bash
pip install transformers torch peft datasets
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print('Libraries loaded!')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

## 1. Prompt Engineering Patterns

Good prompts follow structured patterns. Let's explore the most effective ones.

In [ ]:
# Prompt template patterns
prompt_patterns = {
    'Zero-Shot': {
        'template': 'Classify the sentiment: "{text}"\nSentiment:',
        'example': 'Classify the sentiment: "This product is amazing!"\nSentiment:',
        'when': 'Simple tasks the model already understands',
    },
    'Few-Shot': {
        'template': 'Examples:\n"Great quality" -> Positive\n"Terrible service" -> Negative\n"{text}" ->',
        'example': 'Examples:\n"Great quality" -> Positive\n"Terrible service" -> Negative\n"Decent but pricey" ->',
        'when': 'Tasks needing format guidance or edge cases',
    },
    'Chain-of-Thought': {
        'template': 'Q: {question}\nLet\'s think step by step:\n1.',
        'example': 'Q: If a train travels 60mph for 2.5 hours, how far does it go?\nLet\'s think step by step:\n1.',
        'when': 'Reasoning, math, multi-step logic',
    },
    'Role-Based': {
        'template': 'You are an expert {role}. {task}',
        'example': 'You are an expert data scientist. Explain overfitting to a non-technical stakeholder.',
        'when': 'Specialized knowledge or tone required',
    },
}

for name, info in prompt_patterns.items():
    print(f'=== {name} ===')
    print(f'When to use: {info["when"]}')
    print(f'Example prompt:\n  {info["example"]}')
    print()

## 2. Prompt Structure Analysis

Let's analyze what makes prompts effective by looking at structure components.

In [ ]:
# Anatomy of a well-structured prompt
prompt_components = pd.DataFrame({
    'Component': ['System/Role', 'Context', 'Task/Instruction',
                  'Examples (Few-shot)', 'Output Format', 'Constraints'],
    'Purpose': [
        'Set the persona and expertise level',
        'Provide relevant background information',
        'Clearly state what you want done',
        'Show the desired input-output pattern',
        'Specify JSON, markdown, list, etc.',
        'Length limits, style, what to avoid',
    ],
    'Impact': ['High', 'Medium', 'Critical', 'High', 'High', 'Medium'],
    'Example': [
        'You are a senior ML engineer',
        'Given this dataset with 10K rows...',
        'Classify each review as positive/negative',
        '"Great!" -> Positive',
        'Return as JSON: {"label": ..., "confidence": ...}',
        'Max 100 words, no jargon',
    ],
})
print(prompt_components.to_string(index=False))

## 3. Temperature and Sampling

Temperature controls randomness in LLM outputs. Let's visualize the effect.

In [ ]:
def softmax_with_temperature(logits, temperature=1.0):
    """Apply temperature-scaled softmax."""
    scaled = logits / max(temperature, 1e-8)
    exp_scaled = np.exp(scaled - np.max(scaled))
    return exp_scaled / exp_scaled.sum()

# Simulate token logits for next-word prediction
tokens = ['the', 'a', 'this', 'my', 'our', 'one', 'each', 'some']
logits = np.array([3.0, 2.5, 1.8, 1.2, 0.8, 0.5, 0.2, -0.1])

temperatures = [0.1, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, temp in zip(axes, temperatures):
    probs = softmax_with_temperature(logits, temp)
    bars = ax.bar(tokens, probs, color='steelblue', edgecolor='black')
    bars[0].set_color('coral')  # highlight top token
    ax.set_title(f'T = {temp}', fontsize=12)
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Effect of Temperature on Token Probabilities', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('Low T (0.1) = deterministic/focused | High T (2.0) = creative/random')

## 4. Top-K and Top-P (Nucleus) Sampling

In [ ]:
def top_k_sampling(probs, k):
    """Keep only top-k tokens, re-normalize."""
    top_k_idx = np.argsort(probs)[-k:]
    filtered = np.zeros_like(probs)
    filtered[top_k_idx] = probs[top_k_idx]
    return filtered / filtered.sum()

def top_p_sampling(probs, p):
    """Keep smallest set of tokens whose cumulative prob >= p."""
    sorted_idx = np.argsort(probs)[::-1]
    cumsum = np.cumsum(probs[sorted_idx])
    cutoff = np.searchsorted(cumsum, p) + 1
    keep_idx = sorted_idx[:cutoff]
    filtered = np.zeros_like(probs)
    filtered[keep_idx] = probs[keep_idx]
    return filtered / filtered.sum()

base_probs = softmax_with_temperature(logits, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, probs, title in [
    (axes[0], base_probs, 'Original (T=1.0)'),
    (axes[1], top_k_sampling(base_probs, 3), 'Top-K (K=3)'),
    (axes[2], top_p_sampling(base_probs, 0.9), 'Top-P (P=0.9)'),
]:
    colors = ['coral' if p > 0 else 'lightgray' for p in probs]
    ax.bar(tokens, probs, color=colors, edgecolor='black')
    ax.set_title(title)
    ax.set_ylim(0, 0.6)
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Sampling Strategies', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Real Tokenization with GPT-2

Let's see how a real tokenizer breaks down prompts into tokens.

In [ ]:
# Real tokenization with GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
prompts = [
    "Classify the sentiment: 'This movie was amazing!' Sentiment:",
    "Translate to French: Hello, how are you?",
    "Summarize: The quick brown fox jumps over the lazy dog.",
]

for prompt in prompts:
    tokens = tokenizer.encode(prompt)
    decoded = [tokenizer.decode([t]) for t in tokens]
    print(f"Prompt: {prompt}")
    print(f"  Token IDs: {tokens}")
    print(f"  Tokens: {decoded}")
    print(f"  Count: {len(tokens)}\n")

## 6. Real Text Generation at Different Temperatures

Generate text using distilgpt2 (lightweight, runs on CPU).

In [ ]:
# Text generation with distilgpt2 (small, runs on CPU)
generator = pipeline("text-generation", model="distilgpt2", device=-1)

prompt = "Artificial intelligence will"
print(f"Prompt: '{prompt}'\n")

for temp in [0.3, 0.7, 1.0, 1.5]:
    result = generator(
        prompt,
        max_new_tokens=30,
        temperature=temp,
        do_sample=True,
        top_p=0.9,
        num_return_sequences=1
    )
    print(f"T={temp}: {result[0]['generated_text']}")
    print()

## 7. Real LoRA Configuration with PEFT

Apply Low-Rank Adaptation to dramatically reduce trainable parameters.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Load a small model
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Count original parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Original model parameters: {total_params:,}")

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # rank
    lora_alpha=16,          # scaling factor
    lora_dropout=0.1,
    target_modules=["c_attn"],  # GPT-2 attention layer
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# Compare
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"\nOriginal params: {total_params:,}")
print(f"LoRA trainable params: {trainable:,}")
print(f"Reduction: {(1 - trainable/total_params)*100:.1f}%")

## 8. Parameter Efficiency Visualization

In [ ]:
# Visualize parameter efficiency
d_model = 768  # DistilGPT2 embedding dimension
ranks = [4, 8, 16, 32]

full_finetune_params = total_params
lora_params_by_rank = [d_model * r * 2 for r in ranks]  # A@B matrices

fig, ax = plt.subplots(figsize=(10, 5))
methods = ['Full\nFine-tune'] + [f'LoRA\n(r={r})' for r in ranks]
params = [full_finetune_params] + lora_params_by_rank

colors = ['coral'] + ['steelblue'] * len(ranks)
bars = ax.bar(methods, params, color=colors, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Trainable Parameters', fontsize=11)
ax.set_title('Parameter Efficiency: Full Fine-tune vs LoRA', fontsize=13)
ax.set_yscale('log')

for i, (method, v) in enumerate(zip(methods, params)):
    reduction = (1 - v/full_finetune_params) * 100
    ax.text(i, v * 1.3, f'{v:,}\n({reduction:.0f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Evaluation of Prompts & Fine-Tuned Models

In [ ]:
# Simulate A/B testing prompts
np.random.seed(42)
n_tests = 500

# Prompt A: basic zero-shot
accuracy_a = np.random.binomial(1, 0.72, n_tests)
# Prompt B: few-shot with examples
accuracy_b = np.random.binomial(1, 0.85, n_tests)
# Prompt C: chain-of-thought
accuracy_c = np.random.binomial(1, 0.88, n_tests)

results = pd.DataFrame({
    'Prompt Strategy': ['Zero-Shot', 'Few-Shot (3 examples)', 'Chain-of-Thought'],
    'Accuracy': [accuracy_a.mean(), accuracy_b.mean(), accuracy_c.mean()],
    'Std': [accuracy_a.std()/np.sqrt(n_tests), accuracy_b.std()/np.sqrt(n_tests),
            accuracy_c.std()/np.sqrt(n_tests)],
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(results['Prompt Strategy'], results['Accuracy'],
       yerr=results['Std']*2, color=['#ff9999', '#66b3ff', '#99ff99'],
       edgecolor='black', capsize=5, linewidth=1.5)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.set_title('Prompt Strategy Comparison (with 95% CI)', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Interview Takeaways

**Prompt Engineering:**
- Zero-shot for simple tasks; few-shot for format guidance; CoT for reasoning
- Structure: role + context + task + format + constraints
- Temperature: low (0.0-0.3) for factual, high (0.7-1.0) for creative
- Top-K and Top-P control sampling diversity

**Fine-Tuning:**
- Full fine-tuning updates all weights (most flexible, most expensive)
- **LoRA** adds low-rank matrices — reduces trainable params by 90%+
- **QLoRA** = quantized base model + LoRA (fits on consumer GPUs)
- Always evaluate: accuracy, latency, cost, and alignment
- Prompt engineering is often sufficient before resorting to fine-tuning

**Key Metrics for Evaluation:**
- Accuracy/F1 on task-specific test set
- Latency and throughput
- Parameter and memory footprint
- Cost per inference
- Bias and fairness across demographics

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>